# `index(base, start, end=None, quote=None)`

Daily levels for the market indexes `equity_indexes.prices` holds, from FMP, as stored or in any currency `forex()` can price. Change the arguments and re-run any cell.

> **About the data shown.** This function's data comes from FMP's API, and FMP's license doesn't allow displaying it publicly. So the outputs saved here are synthetic: made-up data from [`_synthetic.py`](_synthetic.py), run through the function's own, unchanged code. The values are not real prices or yields.
>
> To see real data, start Jupyter with `ARORA_REAL_DATA=1`: every cell then reads prod, read-only, as `readonly_user`. Don't commit those outputs; `tests/test_notebooks_offline.py` fails if you save them.

In [1]:
# Synthetic data in place of FMP's (see the note above), unless Jupyter was started with ARORA_REAL_DATA=1.
import _synthetic

_synthetic.use()

Data: synthetic


In [2]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.index import INDEXES, IndexBar
from arora_data.sync import index, list_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The indexes
Read from the table on every call, so an index the main repo adds or drops shows up here with no code change. The 55 the main repo ingests have a name and the currency their members trade in; the 10 volatility series have no currency. Any other stored index works by symbol, with `name` and `currency` blank.

In [3]:
pl.DataFrame([{"index": s.key, "name": INDEXES.get(s.key, (None, None))[0], "currency": INDEXES.get(s.key, (None, None))[1],
               "latest_date": s.latest_date} for s in list_series("equity_indexes", "prices")])

index,name,currency,latest_date
str,str,str,date
"""000001.SS""","""sse composite""","""CNY""",2026-11-23
"""FTSEMIB.MI""","""ftse mib""","""EUR""",2026-11-23
"""^AEX""","""aex""","""EUR""",2026-11-23
"""^AXJO""","""s&p/asx 200""","""AUD""",2026-11-23
"""^BVSP""","""ibovespa""","""BRL""",2026-11-23
…,…,…,…
"""^VIX6M""","""cboe 6-month volatility""",null,2026-11-23
"""^VIX9D""","""cboe 9-day volatility""",null,2026-11-23
"""^VVIX""","""cboe vix volatility""",null,2026-11-23


## By symbol or name
The caret is optional, and a name or alias works anywhere a symbol does, in any case and spacing. Exchange-suffixed symbols such as `000001.SS` have no caret. Aliases include `dow`, `djia`, `tsx`, `taiex`, `asx 200`, `s&p 400` and `shanghai composite`.

In [4]:
week = today_et() - timedelta(days=7)
assert index("^GSPC", week) == index("gspc", week) == index("S&P 500", week)
assert index("dow", week) == index("^DJI", week)
show(index("nikkei 225", week)[-2:] + index("shanghai composite", week)[-2:] + index("vix", week)[-2:])

6 rows

index|name|currency|quote|date|route|quote_date|open|high|low|close|volume|vwap
---|---|---|---|---|---|---|---|---|---|---|---|---
^N225|nikkei 225|JPY|None|2026-11-20|stored|None|97.183629|99.073153|96.507741|98.598211|3714|98.059702
^N225|nikkei 225|JPY|None|2026-11-23|stored|None|98.458575|98.851701|97.167910|97.724663|5367|97.914758
000001.SS|sse composite|CNY|None|2026-11-20|stored|None|82.145664|84.160261|81.550007|84.031353|1675|83.247207
000001.SS|sse composite|CNY|None|2026-11-23|stored|None|83.756593|84.143238|83.309184|83.410452|4386|83.620958
^VIX|cboe volatility|None|None|2026-11-20|stored|None|104.514020|105.826480|104.271896|105.646814|9395|105.248397
^VIX|cboe volatility|None|None|2026-11-23|stored|None|105.367795|106.267367|104.471585|104.852919|4391|105.197290

## Levels as stored, the default
FMP daily bars as stored (`route` = stored): open, high, low, close, volume and VWAP, in the index's own currency (`currency`). `quote` and `quote_date` are blank because no rate is used.

In [5]:
show(index("^GSPC", today_et() - timedelta(days=7)))

6 rows

index|name|currency|quote|date|route|quote_date|open|high|low|close|volume|vwap
---|---|---|---|---|---|---|---|---|---|---|---|---
^GSPC|s&p 500|USD|None|2026-11-16|stored|None|136.006999|136.635752|135.036549|136.217287|1602|135.963196
^GSPC|s&p 500|USD|None|2026-11-17|stored|None|135.820775|138.684961|135.619342|137.501218|9566|137.268507
^GSPC|s&p 500|USD|None|2026-11-18|stored|None|137.017678|137.371069|135.006161|136.032539|7719|136.136590
^GSPC|s&p 500|USD|None|2026-11-19|stored|None|135.498332|137.419132|134.479857|137.393170|8974|136.430720
^GSPC|s&p 500|USD|None|2026-11-20|stored|None|136.134173|137.201392|135.172778|137.199839|1267|136.524669
^GSPC|s&p 500|USD|None|2026-11-23|stored|None|137.308021|138.048860|136.067638|137.790949|8456|137.302483

## In another currency
The stored level times `forex(currency, "USD")` that day (`route` = crossed): here the Nikkei 225 in US dollars. Open uses the day's opening rate, close its closing rate. High, low and VWAP are blank: the index and the rate peak at different moments. Volume stays as stored. A day the pair has no rate carries the last close, and `quote_date` names the rate's day.

In [6]:
show(index("nikkei 225", today_et() - timedelta(days=7), quote="USD"))

6 rows

index|name|currency|quote|date|route|quote_date|open|high|low|close|volume|vwap
---|---|---|---|---|---|---|---|---|---|---|---|---
^N225|nikkei 225|JPY|USD|2026-11-16|crossed|2026-11-16|110.125737152|None|None|110.280658356|8438|None
^N225|nikkei 225|JPY|USD|2026-11-17|crossed|2026-11-17|110.388987536|None|None|108.502891570|1409|None
^N225|nikkei 225|JPY|USD|2026-11-18|crossed|2026-11-18|108.578216603|None|None|109.410086088|1346|None
^N225|nikkei 225|JPY|USD|2026-11-19|crossed|2026-11-19|109.804274057|None|None|110.365447653|9205|None
^N225|nikkei 225|JPY|USD|2026-11-20|crossed|2026-11-20|110.617581142|None|None|113.380930040|3714|None
^N225|nikkei 225|JPY|USD|2026-11-23|crossed|2026-11-23|113.167003629|None|None|112.750596782|5367|None

An index already in the quote currency comes back as stored, so `quote="USD"` puts every index on one footing: the S&P 500 keeps its high, low and VWAP, the others are crossed.

In [7]:
start = today_et() - timedelta(days=7)
rows = [r for q in ["USD", "EUR", "JPY", "GBP", "INR", "CHF"] for r in index("^GSPC", start, quote=q)]
to_df(rows, IndexBar).pivot(on="quote", index="date", values="close")

date,USD,EUR,JPY,GBP,INR,CHF
date,"decimal[38,9]","decimal[38,9]","decimal[38,9]","decimal[38,9]","decimal[38,9]","decimal[38,9]"
2026-11-16,136.217287000,123.106450068,119.715243766,122.063641569,124.775034892,145.483059296
2026-11-17,137.501218000,123.975715401,121.234961412,122.976027489,125.361097962,146.325358665
2026-11-18,136.032539000,122.069278384,119.269521284,121.870132744,124.782784057,145.111078588
2026-11-19,137.393170000,123.170544102,120.421403889,123.123081707,126.436614265,145.842437775
2026-11-20,137.199839000,121.195382733,119.311586791,121.976012838,125.915152242,144.768330919
2026-11-23,137.790949000,120.825904566,119.427962600,123.076753847,125.668652471,146.519730037


In [8]:
world = ["^GSPC", "^FTSE", "^GDAXI", "^N225", "^HSI", "^NSEI", "^BVSP", "000001.SS"]
usd = to_df(index("ALL", today_et() - timedelta(days=7), quote="USD"), IndexBar)
usd.filter(pl.col("index").is_in(world)).pivot(on="index", index="date", values="close").sort("date")

date,000001.SS,^BVSP,^FTSE,^GDAXI,^GSPC,^HSI,^N225,^NSEI
date,"decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]"
2026-11-16,84.0037839757,112.0059325030,115.8063598050,162.4453192180,136.2172870000,98.3522883912,110.2806583560,107.3745382100
2026-11-17,85.3846747662,111.7564622840,116.2160759290,161.1309308660,137.5012180000,96.6252202626,108.5028915700,107.4958983630
2026-11-18,84.7697137684,111.5880684460,114.0541273480,162.1712033760,136.0325390000,96.4408596004,109.4100860880,107.2571456910
2026-11-19,85.7319742823,112.8761816580,113.5127784530,161.4220839710,137.3931700000,95.7357837272,110.3654476530,106.2667654800
2026-11-20,88.3694071995,111.7809256800,112.8297068470,165.6678505340,137.1998390000,96.4808607250,113.3809300400,106.9024876060
2026-11-23,87.5064541040,112.0075600460,109.5955073060,168.1087410070,137.7909490000,97.3689596914,112.7505967820,106.2234162700


## `index("ALL", start, end, quote=None)`
Every stored index, ordered by symbol then date. Each symbol is read once, and each currency's rate once. With a quote, the volatility series (no currency) are left out.

In [9]:
to_df(index("ALL", today_et() - timedelta(days=7)), IndexBar).pivot(on="index", index="date", values="close")

date,000001.SS,FTSEMIB.MI,^AEX,^AXJO,^BVSP,^DJI,^FCHI,^FTSE,^GDAXI,^GSPC,^GSPE,^GSPTSE,^GVZ,^HSI,^IBEX,^IXIC,^KS11,^MID,^MOVE,^MXX,^N225,^NDX,^NSEI,^NYA,^OEX,^OVX,^RLG,^RLV,^RUA,^RUI,^RUT,^RVX,^SP500-15,^SP500-20,^SP500-25,^SP500-30,^SP500-35,^SP500-40,^SP500-45,^SP500-50,^SP500-55,^SP500-60,^SP500TR,^SPXEW,^SSMI,^STOXX,^STOXX50E,^TWII,^VIX,^VIX3M,^VIX6M,^VIX9D,^VVIX,^VXN,^W5000
date,"decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]"
2026-11-16,81.118338,123.830726,84.198724,110.826072,118.094687,74.711601,110.156435,103.773510,146.810049,136.217287,87.291212,101.304007,90.614014,109.666834,88.672486,132.080523,99.437755,108.437140,99.921333,99.249844,96.920708,88.805047,98.355077,91.272935,73.394994,94.868560,107.923769,91.748720,102.380718,86.678358,83.007111,110.212337,101.338992,106.303528,105.544663,95.456093,104.605033,100.280268,92.286133,108.841932,99.514391,111.926315,110.671475,130.190495,81.556575,136.726690,93.805066,128.096131,106.822243,72.396106,111.910336,97.982678,117.904239,83.373858,80.108758
2026-11-17,82.180871,121.976048,84.333593,110.182670,117.013151,73.909852,110.135996,103.939380,145.281058,137.501218,86.775914,101.233173,89.841402,107.806111,88.818202,131.580462,99.636172,109.044731,99.224648,98.380265,95.667108,87.931149,98.004978,89.587540,72.428677,93.783246,107.901399,90.730443,103.201237,85.938356,81.207008,109.904166,100.056488,107.112489,107.199856,95.805575,102.957439,98.571381,91.879022,108.953789,97.668311,111.909616,111.060404,131.040982,82.115083,134.890493,93.167049,128.646397,105.345218,73.099062,111.584328,96.716525,116.318810,82.590648,81.370867
2026-11-18,81.597292,122.294669,83.135163,109.085188,116.567686,74.034653,109.270531,102.179903,145.524901,136.032539,86.627968,101.076222,89.626343,107.974222,88.449056,130.563915,98.499731,107.866665,100.872475,97.383326,95.927700,88.029147,98.387087,91.068285,72.956121,93.050734,108.947002,91.259022,105.234295,86.109528,82.061677,110.760968,101.097494,106.813834,106.020965,95.947112,102.506939,98.776690,92.661215,111.011564,98.087012,111.312620,111.901457,132.331255,82.226601,132.726123,93.140560,130.162951,105.632064,72.250273,112.413931,96.066279,118.092976,84.364924,80.916944
2026-11-19,81.779816,121.760187,83.199826,109.465495,118.258118,74.562501,108.088079,101.722983,144.712040,137.393170,85.002672,101.832812,89.132877,107.446472,89.466809,128.270592,95.640687,107.545519,99.855975,98.704431,96.732335,88.181539,97.792416,88.453933,73.161221,92.022635,109.542016,91.644672,101.810175,86.245297,80.768325,111.627175,102.666790,107.564009,105.098657,96.633321,102.495012,98.889850,91.481809,110.415910,98.903730,111.427011,109.079314,132.738187,81.734919,133.769187,90.790987,130.310815,105.064484,72.510693,109.701207,96.240018,116.336328,84.957738,82.751654
2026-11-20,84.031353,120.630982,83.261598,109.313862,117.217391,74.705138,106.910586,100.310014,146.342581,137.199839,84.328960,100.733949,88.632159,108.066186,89.647260,128.152776,95.786089,110.053401,99.980230,98.697874,98.598211,88.513817,98.109758,87.694937,72.778755,91.981749,108.041252,91.472649,101.812580,85.961143,79.778574,109.929478,102.772566,106.916681,105.238927,9

## Full history in one call
The S&P 500 from the start of FMP's history. There is no row limit.

In [10]:
spx = to_df(index("^GSPC", date(1900, 1, 1)), IndexBar)
print(f"{spx.height:,} days, {spx['date'].min()} to {spx['date'].max()}")
(
    spx.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

25,800 days, 1928-01-03 to 2026-11-23


year,last_day,close
i32,date,"decimal[38,6]"
2017,2017-12-29,102.158348
2018,2018-12-31,82.122721
2019,2019-12-31,103.464622
2020,2020-12-31,103.060163
2021,2021-12-31,93.107695
2022,2022-12-30,103.495895
2023,2023-12-29,85.302751
2024,2024-12-31,110.380521
2025,2025-12-31,126.004502


## Duplicates FMP lists
`^SPX` is the same series as `^GSPC`, so it returns `^GSPC`. `^DJX`, `^XSP`, `^NANOS` and `^MRUT` are scaled copies of `^DJI`, `^GSPC` and `^RUT`; they are not stored and raise an error naming the index to use.

In [11]:
assert index("^SPX", week) == index("^GSPC", week)
for base in ["^DJX", "xsp", "^NANOS", "mrut"]:
    try:
        index(base, week)
    except ValueError as err:
        print(err)

^DJX is a scaled copy of ^DJI and is not stored: use ^DJI
^XSP is a scaled copy of ^GSPC and is not stored: use ^GSPC
^NANOS is a scaled copy of ^GSPC and is not stored: use ^GSPC
^MRUT is a scaled copy of ^RUT and is not stored: use ^RUT


## Bad input
A symbol FMP lists but the main repo doesn't ingest, a partial name, a quote for a volatility series (it has no currency), `quote="ALL"`, a currency `forex()` can't price, or a start after the end each raise a clear error.

In [12]:
for base, quote in [("^TNX", None), ("^SOX", None), ("s&p", None), ("nasdaq", None), ("vix", "EUR"), ("^GSPC", "ALL"),
                    ("^GSPC", "XYZ")]:
    try:
        index(base, week, quote=quote)
    except ValueError as err:
        print(f"{base}/{quote}: {err}")
try:
    index("^GSPC", today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")

^TNX/None: ^TNX is not a stored index: use a symbol or name, such as ^GSPC, GSPC or s&p 500
^SOX/None: ^SOX is not a stored index: use a symbol or name, such as ^GSPC, GSPC or s&p 500
s&p/None: ^S&P is not a stored index: use a symbol or name, such as ^GSPC, GSPC or s&p 500
nasdaq/None: ^NASDAQ is not a stored index: use a symbol or name, such as ^GSPC, GSPC or s&p 500
vix/EUR: ^VIX has no currency to convert from: leave quote out
^GSPC/ALL: quote must be one currency: "ALL" is for base only
^GSPC/XYZ: XYZ has no stored USD pair, so USD/XYZ can't be computed
start after end: start must be on or before end
